In [ ]:
import json
from pathlib import Path
import numpy as np
from tqdm import tqdm

In [ ]:
def get_pairwise_similarity(embeddings):
    # Convert to numpy array
    embeddings = np.asarray(embeddings)
    
    # Compute norms for all embeddings at once
    norms = np.linalg.norm(embeddings, axis=1)
    
    # Compute dot products between all pairs
    dot_products = np.dot(embeddings, embeddings.T)
    
    # Compute similarities matrix
    similarities = dot_products / np.outer(norms, norms)
    
    # Extract upper triangle (excluding diagonal) to match original output format
    return similarities[np.triu_indices_from(similarities, k=1)]

In [ ]:
# Adapter für fixes Format [ [[vec],...], [[vec],...], ... ]

def load_embedding_data(file_path: str):
    path = Path(file_path)
    with path.open("r", encoding="utf-8") as f:
        raw_data = json.load(f)
    return [{"embeddings": question_embeddings} for question_embeddings in raw_data]

In [ ]:
# Intramodell-Homogenität berechnen

def compute_intramodel_similarity(file_path: str):
    data = load_embedding_data(file_path)

    similarities_per_question = []
    for d in tqdm(data, desc=f"Processing {Path(file_path).name}"):

        d_embeddings = d["embeddings"]

        similarities = get_pairwise_similarity(d_embeddings)
        avg_similarity = np.array(similarities).mean()
        similarities_per_question.append(float(avg_similarity))

    return similarities_per_question

In [ ]:
# Intramodell-Heatmap
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

def model_label_from_file(path_str: str):
    name = Path(path_str).stem
    if name.startswith("embeddings_"):
        name = name[len("embeddings_"):]
    return name

def plot_intramodel_heatmap(file_paths, bins_count=10):
    all_model_data = []
    model_names_used = []

    for model_file in file_paths:
        model_vals = compute_intramodel_similarity(file_path=model_file)

        counts, bins = np.histogram(model_vals, bins=bins_count, range=(0, 1))
        percentages = counts / len(model_vals) * 100

        all_model_data.append(percentages)
        model_names_used.append(model_label_from_file(model_file))

    heatmap_data = np.array(all_model_data).T

    avg_column = np.mean(heatmap_data, axis=1, keepdims=True)
    heatmap_data = np.hstack((avg_column, heatmap_data))

    bucket_labels = [f"{bins[i]:.1f}-{bins[i+1]:.1f}" for i in range(len(bins) - 1)]
    bucket_labels = bucket_labels[::-1]
    heatmap_data = np.flipud(heatmap_data)

    model_names_with_avg = ["Average"] + model_names_used

    fig_width = max(10, 1.0 + 0.55 * len(model_names_with_avg))
    plt.figure(figsize=(fig_width, 6))

    ax = sns.heatmap(
        heatmap_data,
        annot=True,
        fmt=".1f",
        annot_kws={"size": 10},
        cmap="YlOrRd",
        xticklabels=model_names_with_avg,
        yticklabels=bucket_labels,
        cbar=False
    )

    plt.ylabel("Similarity Score Ranges", fontsize=13)
    ax.xaxis.set_ticks_position("top")
    ax.xaxis.set_label_position("top")
    plt.xticks(rotation=45, ha="left", fontsize=11)
    plt.yticks(rotation=0, fontsize=11)

    for tick in ax.get_xticklabels():
        tick.set_x(tick.get_position()[0] - 0.02)

    plt.tight_layout()
    plt.show()

# alle Embedding-Dateien einsammeln
all_model_files = sorted([str(p) for p in Path("./replikation").glob("embeddings_*.json")])
#all_model_files = sorted([str(p) for p in Path("./eigenanteil").glob("embeddings_*.json")])

print("Gefundene Dateien:", len(all_model_files))
for p in all_model_files:
    print("-", p)

plot_intramodel_heatmap(all_model_files, bins_count=10)